In [ ]:
import os
import re
import json
from datasets import Dataset
import tqdm


def parse_full_reasoning(full_reasoning):
    reasoning = []
    for node in full_reasoning:
        if node.get("node_type") not in ['REPHASE_QUESTION', 'FINAL_ANSWER', 'USER_QUESTION']:
            reasoning.append(node.get("node_content", ""))
    reasoning = [f"Step {i+1}: {step}\n" for i, step in enumerate(reasoning)]
    if full_reasoning[-1].get("node_type") == "FINAL_ANSWER":
        final_answer = f"{full_reasoning[-1].get('detailed_answer', '')}\nFinal answer: {full_reasoning[-1].get('node_content', '')}"
        reasoning.append(final_answer)
    return '\n'.join(reasoning)

# get all .jsonl files in the current directory
path = "mcts_data"
jsonl_files = [f for f in os.listdir(path) if f.endswith('.jsonl')]
# jsonl_files = jsonl_files[:2]  # Limit to the first 2 files for testing
data = []
for file in tqdm.tqdm(jsonl_files, desc="Processing files"):
    # Extract the index from the filename with the format "mcts_tree_ 2wiki_<id>.jsonl"
    # example: "mcts_tree_ 2wiki_dev_45.jsonl" ==> id = "dev_45"
    idx = re.search(r'mcts_tree_ 2wiki_(\w+_\d+).jsonl', file)
    if idx:
        idx = idx.group(1)
    else:
        print(f"Could not extract index from filename: {file}")
        continue
    full_path = os.path.join(path, file)
    # Read the jsonl file, decode each line into a dictionary
    all_answers = []
    user_question = ""
    with open(full_path, 'r') as f:
        for line in f:
            try:
                data_dict = json.loads(line)
                if data_dict.get("node_type") == "FINAL_ANSWER":
                    if user_question == "":
                        user_question = data_dict.get("user_question", "")
                    answer = data_dict.get("node_content", "")
                    detailed_answer = data_dict.get("detailed_answer", "")
                    reasoning_path = data_dict.get("reasoning_path", "")
                    full_reasoning = data_dict.get("full_reasoning_path", [])
                    reasoning = parse_full_reasoning(full_reasoning)
                    all_answers.append({
                        # 'user_question': user_question,
                        'answer': answer,
                        'detailed_answer': detailed_answer,
                        'reasoning_path': reasoning_path,
                        'full_reasoning': reasoning
                    })
            except:
                print(f"Error decoding JSON in file {file}: {line}")
                continue
    data.append({
        'id': idx,
        'question': user_question,
        'answers': all_answers,
    })

# Convert data into dataset format
dataset = Dataset.from_list(data)


In [ ]:
from agents.roles.evaluator import Evaluator


online_model_kwargs = {
        'model_name': 'openai/qwen3-8B', 
        'url': 'http://ip-10-4-226-205:30000/v1', 
        'api_key': 'your_api_key_here',  # Replace with your actual API key
        'client_type': 'openai',  # Use 'litellm' for LiteLLMClient or 'openai' for OpenAIClient
        'concurrency': 64,
    }
eval_kwargs = {
        # For creative tasks (creative writing) set it ~ 1, 
        # For logical or factual tasks (summarization, coding, analysis) set it ~ 0
        # For general conversation set it ~ 0.7
        'temperature': 0.1,  
        'n': 5, 
        'top_p': 0.9,
        'max_tokens': 1024*8,  # Set to a high value to allow for long responses
        # Want more varied responses (alongside high temperature) set top_k to 50 - 100 
        # For greedy decoding set it to 1
        'top_k': 20,
        'tensor_parallel_size': 1,
        'reasoning_effort': 'medium',  # Set to 'high'/'medium'/'low' for using thinking capabilities
    }
evaluator = Evaluator(
    client_kwargs=online_model_kwargs, 
    generate_kwargs=eval_kwargs, 
    # verbose=True,
    use_cache=True, 
    cache_dir="mcts_cache/evaluator_cache",
)


def generate_final_answer(example, evaluator):
    question = example['question']
    answers = example['answers']
    final_answer = None
    final_reasoning = None
    if len(answers) > 0:
        reasoning_paths = [answer['full_reasoning'] for answer in answers]
        detailed_answers = [answer['detailed_answer'] for answer in answers if answer['detailed_answer']]
        try:
            final_answer, final_reasoning = evaluator.synthesize_final_answer(question=user_question, reasoning_paths=reasoning_paths)
        except:
            print(f"Error synthesizing final answer for question: {question}")
            print(f"Reasoning paths: {reasoning_paths}")
            print(f"Number of candidates: {len(reasoning_paths)}")
            final_answer, final_reasoning = evaluator.synthesize_final_answer(question=user_question, reasoning_paths=detailed_answers)
            
    return {
        'id': example['id'],
        'question': question,
        'pred': final_answer,
        'detailed_answer': final_reasoning,
        'all_candidates_answers': reasoning_paths
    }

# Generate final answers for each example in the dataset
dataset = dataset.map(
    lambda example: generate_final_answer(example, evaluator),
    # num_proc=512,  # Adjust based on your system's capabilities
    desc="Generating final answers",
    remove_columns=dataset.column_names,
    num_proc=512
)

# Save the dataset
dataset.save_to_disk("results/mcts/wiki2/4B-retriever-8B-generators")

In [ ]:
# Get the mapping the dataset into the original dataset
import datasets

origin_dataset = datasets.load_dataset('RUC-NLPIR/FlashRAG_datasets', '2wikimultihopqa', split='dev')

def get_the_golden_answer(example, origin_dataset):
    # Find the item in the original dataset with the same id
    item = origin_dataset.filter(lambda x: x['id'] == example['id'])
    assert len(item) >= 1, f"Item with id {example['id']} not found in the original dataset"
    return {
        'golden_answers': item[0]['golden_answers'],
    }

dataset = dataset.map(
    lambda example: get_the_golden_answer(example, origin_dataset),
    # num_proc=512,  # Adjust based on your system's capabilities
    desc="Getting the golden answers",
    num_proc=512
)

In [ ]:
import litellm

kwargs = {
    'model': 'bedrock/us.anthropic.claude-3-7-sonnet-20250219-v1:0',
    'aws_profile_name': 'hieu'
}

response = litellm.completion(
    messages=[{'role': 'user', 'content': 'What is your name?'}],
    **kwargs
)

In [ ]:
import datasets
from preprocess.utils import simple_preprocess


def merge_chunks(chunks, chunk_size=512):
    """
    Merge chunks into a single chunk if the total size is less than chunk_size.
    """
    merged_chunks = []
    current_chunk = ""
    current_chunk_size = 0
    for chunk in chunks:
        chunk_size_words = len(chunk.split())
        if current_chunk_size + chunk_size_words <= chunk_size:
            current_chunk += chunk + "\n"
            current_chunk_size += chunk_size_words
        else:
            if current_chunk:
                merged_chunks.append(current_chunk.strip())
            current_chunk = chunk + "\n"
            current_chunk_size = chunk_size_words
    if current_chunk:
        if len(current_chunk.split()) < 128:  # If the last chunk is smaller than 128 words, merge it with the previous chunk
            if merged_chunks:
                merged_chunks[-1] += "\n" + current_chunk.strip()
            else:
                # If there are no previous chunks, just add the current chunk
                merged_chunks.append(current_chunk.strip())
        else:
            merged_chunks.append(current_chunk.strip())
    return merged_chunks
    

def chunk_document(example, chunk_size=512):
    """
    Chunk the document into smaller parts. 
    First chunking the document into paragraphs. Then, if the paragraph is smaller than the chunk size, merge it with the next paragraph.
    """
    # batched must be True to process the dataset in batches with bsize = 1
    document = example['text'][0]
    title = example['title'][0]
    title = simple_preprocess(title)
    paragraphs = document.split('\n\n')
    # Remove empty paragraphs
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    paragraphs = [simple_preprocess(p) for p in paragraphs]
    # Remove paragraphs that are empty after preprocessing
    paragraphs = [p for p in paragraphs if p]
    chunks = merge_chunks(paragraphs, chunk_size=chunk_size)
    # Preprocess each chunk
    all_chunks = []
    for chunk in chunks:
        if len(chunk.split()) > chunk_size*2:
            sub_chunks = chunk.split('\n')
            sub_chunks = [s for s in sub_chunks if s.strip()]
            sub_chunks = merge_chunks(sub_chunks, chunk_size=chunk_size)
            # if the sub_chunk is still larger than chunk_size, split it into smaller chunks by using word splitting
            all_sub_chunks = []
            for sub_chunk in sub_chunks:
                if len(sub_chunk.split()) > chunk_size*2:
                    words = sub_chunk.split()
                    for i in range(0, len(words), chunk_size):
                        all_sub_chunks.append(' '.join(words[i:i + chunk_size]))
                else:
                    all_sub_chunks.append(sub_chunk)
            last_item = all_sub_chunks.pop()
            if len(last_item.split()) < 128:  # If the last sub_chunk is smaller than 128 words, merge it with the previous sub_chunk
                if all_sub_chunks:
                    all_sub_chunks[-1] += f" {last_item}"
                else:
                    # If there are no previous sub_chunks, just add the last item
                    all_sub_chunks.append(last_item)
            else:
                all_sub_chunks.append(last_item)
            all_chunks.extend(all_sub_chunks)
        else:
            all_chunks.append(chunk)
    # remove short chunks
    all_chunks = [chunk for chunk in all_chunks if len(chunk.split()) >= 32]  # Minimum 32 words per chunk
    all_chunks = [f"{title}\n{chunk}" for chunk in all_chunks]
    return  {
        'id': example['id'] * len(all_chunks),
        'contents': all_chunks,
        'title': [title] * len(all_chunks),
        'url': example['url'] * len(all_chunks),
    }
    

dataset = datasets.load_dataset("wikimedia/wikipedia", "20231101.en", split="train")
dataset = dataset.map(
    chunk_document,
    batched=True,
    batch_size=1,  # Process one document at a time
    desc="Chunking documents",
    num_proc=32,
    remove_columns=dataset.column_names,
)
# Filter out documents that have no contents or too long contents
dataset = dataset.filter(lambda x: len(x['contents']) > 0 and len(x['contents']) < 10000,
                        num_proc=32,
                        desc="Filtering documents with no contents or too long contents")
dataset.save_to_disk("data/wiki23-chunked")


/fsx/ubuntu/users/hieuman/miniconda/envs/rag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Calculating length of each chunk (num_proc=32): 100%|██████████| 9677787/9677787 [00:57<00:00, 169188.00 examples/s]


In [1]:
import datasets


dataset = datasets.load_from_disk("data/wiki23-chunked-1")
# Filter out documents that have no contents or too long contents
dataset = dataset.filter(lambda x: len(x['contents']) > 0 and len(x['contents']) < 10000,
                        num_proc=32,
                        desc="Filtering documents with no contents or too long contents")
dataset.save_to_disk("data/wiki23-chunked")

/fsx/ubuntu/users/hieuman/miniconda/envs/rag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Filtering documents with no contents or too long contents (num_proc=32): 100%|██████████| 9677787/9677787 [00:18<00:00, 532212.03 examples/s]
Saving the dataset (41/41 shards): 100%|██████████| 9677704/9677704 [01:26<00:00, 111956.11 examples/s]


In [2]:
import datasets


dataset = datasets.load_from_disk("data/wiki23-chunked")

# Statistics of the dataset length

dataset_len = dataset.map(
    lambda example: {'length': len(example['contents'].split())},
    num_proc=32,
    desc="Calculating length of each chunk",
)

dataset_char_len = dataset.map(
    lambda example: {'length': len(example['contents'])},
    num_proc=32,
    desc="Calculating character length of each chunk",
)

print("Statistics of the dataset length:")
lengths = dataset_len['length']
print(f"Total number of chunks: {len(lengths)}")
print(f"Average length of chunks: {sum(lengths) / len(lengths):.2f} words")
print(f"Minimum length of chunks: {min(lengths)} words")
print(f"Maximum length of chunks: {max(lengths)} words")    

print("Statistics of the dataset character length:")
char_lengths = dataset_char_len['length']
print("Average character length of chunks: {:.2f} characters".format(sum(char_lengths) / len(char_lengths)))
print("Minimum character length of chunks: {} characters".format(min(char_lengths)))
print("Maximum character length of chunks: {} characters".format(max(char_lengths)))

Calculating length of each chunk (num_proc=32): 100%|██████████| 9677704/9677704 [00:40<00:00, 237283.16 examples/s]
Calculating character length of each chunk (num_proc=32): 100%|██████████| 9677704/9677704 [00:22<00:00, 428081.77 examples/s]


Statistics of the dataset length:
Total number of chunks: 9677704
Average length of chunks: 319.86 words
Minimum length of chunks: 32 words
Maximum length of chunks: 1036 words
Statistics of the dataset character length:
Average character length of chunks: 2013.80 characters
Minimum character length of chunks: 128 characters
Maximum character length of chunks: 9989 characters


In [ ]:
import json
import os
import tqdm
import datasets


datapath = "extractor_data/2wiki/extractor-e5-Claude-3.7-sonnet/bedrock/us.anthropic.claude-3-7-sonnet-20250219-v1:0"
# Read all .json files in the directory
files = [f for f in os.listdir(datapath) if f.endswith('.json')]
all_data = []
for file in tqdm.tqdm(files, desc="Processing files"):
    full_path = os.path.join(datapath, file)
    # Read the json file, decode it into a dictionary
    with open(full_path, 'r') as f:
        data = json.load(f)
        for raw_data in data:
            item = {
                'user_question': raw_data['any_other_info']['user_question'],
                'user_question_id': raw_data['any_other_info']['question_id'],
                'depth': raw_data['any_other_info']['depth'],
                'question': raw_data['any_other_info']['question'],
                'input': raw_data['input'],
                'output': raw_data['output'],
                'reasoning': raw_data['reasoning'],
            }
            if 'retrieved_docs' in raw_data['any_other_info']:
                item_type = 'explore'
                content = raw_data['any_other_info']['retrieved_docs']
            elif 'memory_knowledge' in raw_data['any_other_info']:
                item_type = 'reflect'
                content = raw_data['any_other_info']['memory_knowledge']
            elif 'raw_memory' in raw_data['any_other_info']:
                item_type = 'synthesize'
                content = raw_data['any_other_info']['raw_memory']
            else:
                raise ValueError(f"Unknown item type in file {file}: {raw_data['any_other_info']}")
            item['type'] = item_type
            item['content'] = content
            all_data.append(item)

# Convert all_data into a dataset
